In [1]:
from backend.app.utils.get_data import get_fyers_authcode, get_historical_data_by_fyers, save_to_csv

# get_fyers_authcode()


In [2]:
securities = {
    # Equity Stocks
    "HDFC BANK LTD": ("NSE:HDFCBANK-EQ", "HDFCBANK.csv"),
    "TATA CONSULTANCY SERVICES": ("NSE:TCS-EQ", "TCS.csv"),
    "RELIANCE INDUSTRIES LTD": ("NSE:RELIANCE-EQ", "RELIANCE.csv"),
    "INFOSYS LIMITED": ("NSE:INFY-EQ", "INFY.csv"),
    "HINDUSTAN UNILEVER LTD": ("NSE:HINDUNILVR-EQ", "HINDUNILVR.csv"),

    # Indexes
    "NIFTY 50": ("NSE:NIFTY50-INDEX", "NIFTY50.csv"),
    "NIFTY BANK": ("NSE:NIFTYBANK-INDEX", "NIFTYBANK.csv"),
    "NIFTY IT": ("NSE:NIFTYIT-INDEX", "NIFTYIT.csv"),
    "NIFTY MIDCAP 100": ("NSE:NIFTYMIDCAP100-INDEX", "NIFTYMIDCAP100.csv"),
    "NIFTY FMCG": ("NSE:NIFTYFMCG-INDEX", "NIFTYFMCG.csv"),
}

In [4]:
symbol= securities["RELIANCE INDUSTRIES LTD"][0]
df = get_historical_data_by_fyers(symbol=symbol,
                                  resolution='30',
                                  start_date="01-11-2024",
                                  end_date="01-11-2025")
df.shape

{'candles': [[1730463300, 1333.05, 1341.95, 1333, 1338.55, 931147], [1730465100, 1338.5, 1340.85, 1338, 1338.2, 809484], [1730466900, 1338.3, 1338.55, 1337, 1338.4, 384956], [1730691900, 1337.85, 1340, 1304.1, 1305.15, 4330502], [1730693700, 1304.8, 1305.65, 1293.25, 1293.95, 2078185], [1730695500, 1293.8, 1296.25, 1288.35, 1291.4, 1616451], [1730697300, 1291.2, 1293.75, 1285.1, 1287.65, 1325440], [1730699100, 1287.65, 1295.75, 1285.25, 1294.6, 1052978], [1730700900, 1294.6, 1297, 1290, 1295.35, 848947], [1730702700, 1295.3, 1298.7, 1295, 1297.85, 1378064], [1730704500, 1297.75, 1298.7, 1294, 1294.4, 1021846], [1730706300, 1294.25, 1295.45, 1290, 1292.65, 754572], [1730708100, 1292.65, 1296.4, 1291.35, 1295.9, 657936], [1730709900, 1296, 1299.45, 1293.5, 1297.65, 878222], [1730711700, 1297.85, 1308.45, 1297.5, 1300.7, 2268708], [1730713500, 1300.8, 1301.35, 1297, 1298.85, 1577694], [1730778300, 1293, 1298.75, 1286.15, 1293.3, 3414950], [1730780100, 1293.3, 1301.25, 1292.55, 1299.55, 11

(3204, 6)

In [11]:
import vectorbt as vbt
prices = df['close']

# ------------------------------------------------------------
# 2. Build EMA indicators (9 & 26)
# ------------------------------------------------------------
ema_fast = vbt.MA.run(prices, window=9)
ema_slow = vbt.MA.run(prices, window=26)

# ------------------------------------------------------------
# 3. Define Entry/Exit Signals
# ------------------------------------------------------------
entries = ema_fast.ma_crossed_above(ema_slow)
exits   = ema_fast.ma_crossed_below(ema_slow)

# ------------------------------------------------------------
# 4. Run Backtest
# ------------------------------------------------------------
portfolio = vbt.Portfolio.from_signals(
    close=prices,
    entries=entries,
    exits=exits,
    init_cash=10000,
    fees=0.0005,
    slippage=0.0,
    freq='30min'
)


In [12]:
portfolio.stats()


Start                                        0
End                                       3203
Period                        66 days 18:00:00
Start Value                            10000.0
End Value                         10907.935108
Total Return [%]                      9.079351
Benchmark Return [%]                 11.090359
Max Gross Exposure [%]                   100.0
Total Fees Paid                     673.677726
Max Drawdown [%]                     14.207412
Max Drawdown Duration         33 days 21:30:00
Total Trades                                64
Total Closed Trades                         64
Total Open Trades                            0
Open Trade PnL                             0.0
Win Rate [%]                              37.5
Best Trade [%]                        5.785381
Worst Trade [%]                      -4.282663
Avg Winning Trade [%]                 2.087434
Avg Losing Trade [%]                 -1.006071
Avg Winning Trade Duration     0 days 21:47:30
Avg Losing Tr

In [ ]:


# ------------------------------------------------------------
# 5. Print All Performance Metrics
# ------------------------------------------------------------
print("\n===== PERFORMANCE METRICS =====\n")
print(portfolio.stats())

# ------------------------------------------------------------
# 6. Interactive Trading Chart (Better Than TradingView)
# ------------------------------------------------------------
fig = go.Figure()

# --- Candlestick Chart
fig.add_trace(go.Candlestick(
    x=df.index,
    open=df['open'],
    high=df['high'],
    low=df['low'],
    close=df['close'],
    name="Price"
))

# --- EMAs
fig.add_trace(go.Scatter(
    x=df.index, y=ema_fast.ma, 
    mode="lines", name="EMA 9"
))
fig.add_trace(go.Scatter(
    x=df.index, y=ema_slow.ma, 
    mode="lines", name="EMA 26"
))

# --- Buy Signals
fig.add_trace(go.Scatter(
    x=df.index[entries.values],
    y=prices[entries.values],
    mode='markers',
    marker=dict(symbol='triangle-up', size=12),
    name="BUY"
))

# --- Sell Signals
fig.add_trace(go.Scatter(
    x=df.index[exits.values],
    y=prices[exits.values],
    mode='markers',
    marker=dict(symbol='triangle-down', size=12),
    name="SELL"
))

fig.update_layout(
    title="EMA 9 / 26 Crossover Backtest",
    xaxis_title="Time",
    yaxis_title="Price",
    xaxis_rangeslider_visible=False,
    height=700
)


fig.show()

# ------------------------------------------------------------
# 7. Equity Curve & Drawdown (Interactive)
# ------------------------------------------------------------
portfolio.plot().show()